In [ ]:
import pydocmaker as pyd
import os
from pathlib import Path

## Using templates (HTML and LaTeX)

pydocmaker supports mounting Jinja2 Templates organized in folders together with (optional) default parameters and assets. Such templates are organized in folders. `pydocmaker` has one such template directory with a `base` template shipped with it by default which is part of the installation.


### The `pydocmaker` inbuild templates

In [ ]:
pyd.get_registered_template_dirs()

In [ ]:
pyd.get_available_template_ids()

### Definition of templates

Each template has 
- a name, and its format. The main name per template is then: `<name>.<format>.j2`
- Optionally it can also have an assests folder `<name>.assets/` where you can keep any assests such as images/logos etc. 
- Optionally it can have a JSON file `<name>.params.json` to define the parameters which can be used for this template type, since (as far as I know) Jinja2 has no possibility to reflect / track expected parameter names. 

Such templates only work for text based formats, such as `html` and `tex`. 

In [ ]:
for d in pyd.get_registered_template_dirs():
    print(d)
    for f in os.listdir(d):
        print('   ', f)

        

### Adding / defining you own template

Such Suppose you have the following folder structure 

```
home/jovyan/templates/
├─ assets/
│  ├─ i_can_use_this_everywhere.png
├─ fancy_tempate.assets/
│  ├─ fancy_logo.png
│  ├─ fancy_title_picture.png
├─ fancy_template.params.json
├─ fancy_template.tex.j2
├─ normal_template.params.json
├─ normal_template.tex.j2
```

(NOTE: you can also check out the `templates` folder in this repository for an example).

You can then mount this folder in pydocmaker using 

In [ ]:
p = (Path(pyd.__file__).root.root / 'example_templates')
print(p)
pyd.register_new_template_dir(p)

Which will give you two available templates to use for exporting tex and pdf documents: 


In [ ]:
pyd.get_available_template_ids()

### Using your template
You can mark the templates to be used for a doc by setting it to the reports metadata:

In [ ]:
doc = pyd.get_example()
doc.set_template_to_meta('fancy_template')


which will write all needed data to the documents "metadata". Specifically:
- `template_id` will hold the id with which the specific template can be loaded. In our case it will be `fancy_template`.
- `files_to_upload` will hold all assets as base64 encoded bytes in our case the following files:
   - **key**: `fancy_logo.png` **value** content from `.../fancy_tempate.assets/fancy_logo.png`
   - **key**: `fancy_title_picture.png` **value** content from `.../fancy_tempate.assets/fancy_title_picture.png` 
    - **key**: `i_can_use_this_everywhere.png` **value** content from `.../assets/i_can_use_this_everywhere.png` (content from `assets` will be made available shared for all templates)
- and all other fields loaded from `fancy_template.params.json` will be loaded to the metadata dictionary directly. 

If you thereafter export your document to pdf (or html if you have an html type template), pydocmaker will automatically load the template and render it with parameters, attachments and your document as the body. 

you can view the template and params by 

In [ ]:
doc.get_meta()

In [ ]:
doc.update_meta(author='Me!')

### Jinja2 templates from strings


you can also use Jinja2 Templates directly to make HTML or PDF (latex) documents. An example is given below:


In [ ]:
# This is a minimal template, the document will be written to the "body" part.
template = r'''
\documentclass[a4paper]{article}

{% if title %}\title {{ title }}{% endif %}
{% if author %}\author {{ author }}{% endif %}

\begin{document}

{{ body }}

\end{document}
'''

from jinja2 import Environment, FileSystemLoader
import pydocmaker as pyd


doc = pyd.get_example()
pdf_bytes = doc.to_pdf(template=template, template_params=dict(title='My Title', author='Me'))
len(pdf_bytes), type(pdf_bytes)

In [ ]:
tex_string, attachments = doc.to_tex(template=template, template_params=dict(title='My Title', author='Me'))
print(tex_string[:100] + '...')


**NOTE**: If you use template strings directly and your document template has external references such as logos, you need to load them to a bytes array and pass them as a `{filename:content}` dictionary into the to_pdf(...) methods using the `files_to_upload` argument.


In [ ]:
assets = {}
with open(r'C:\Users\tglaubach\repos\pydocmaker\example_templates\fancy_template.assets\fancy_logo.png', 'rb') as fp:
    assets = {'fancy_logo.png': fp.read()} 

pdf_bytes = doc.to_pdf(template=template, template_params=dict(title='My Title', author='Me'), files_to_upload=assets)
len(pdf_bytes), type(pdf_bytes)